In [59]:
!pip install -q pymupdf pandas tqdm

In [46]:
import fitz
import pandas as pd
import os

from google.colab import files

In [47]:
import os

pdf_files = [
    f for f in os.listdir("/content")
    if f.endswith(".pdf")
]

print("PDFs Found:")
print(pdf_files)

PDFs Found:
['amt_powerplant_handbook.pdf', 'amt_general_handbook.pdf', 'FAA-H-8083-31B_Aviation_Maintenance_Technician_Handbook.pdf']


In [ ]:
all_documents = []

for pdf in pdf_files:

    print(f"Reading {pdf}...")

    doc = fitz.open(pdf)

    text = ""

    for page in doc:
        text += page.get_text()

    all_documents.append({
        "filename": pdf,
        "text": text
    })

print("\nFinished Reading All PDFs!")

Reading amt_powerplant_handbook.pdf...
Reading amt_general_handbook.pdf...


In [ ]:
for doc in all_documents:

    print("="*80)

    print(doc["filename"])

    print("Characters:", len(doc["text"]))

    print(doc["text"][:300])

    print()

In [ ]:
def split_text(text, chunk_size=700, overlap=150):

    chunks = []

    start = 0

    while start < len(text):

        end = start + chunk_size

        chunks.append(text[start:end])

        start += chunk_size - overlap

    return chunks

In [ ]:
chunk_data = []

for document in all_documents:

    chunks = split_text(document["text"])

    for i, chunk in enumerate(chunks):

        chunk_data.append({
            "source": document["filename"],
            "chunk_id": i,
            "text": chunk
        })

print("Total Chunks:", len(chunk_data))

In [ ]:
print(pdf_files)
print(len(pdf_files))

In [ ]:
print("Number of documents:", len(all_documents))

for doc in all_documents:
    print(doc["filename"], "->", len(doc["text"]))

In [ ]:
import os

for pdf in pdf_files:
    print(pdf)
    print(os.path.getsize(pdf), "bytes")
    print("-"*50)

In [ ]:
for pdf in pdf_files:
    with open(pdf, "rb") as f:
        print(pdf)
        print(f.read(20))
        print()

In [61]:
import fitz
import os

all_documents = []

for pdf in pdf_files:

    pdf_path = os.path.join("/content", pdf)

    print(f"Reading {pdf_path}")

    doc = fitz.open(pdf_path)

    text = ""

    for page in doc:
        text += page.get_text("text")

    print("Characters extracted:", len(text))

    all_documents.append({
        "filename": pdf,
        "text": text
    })

print("\nAll PDFs loaded successfully!")

Reading /content/amt_powerplant_handbook.pdf
Characters extracted: 1830419
Reading /content/amt_general_handbook.pdf
Characters extracted: 2381774
Reading /content/FAA-H-8083-31B_Aviation_Maintenance_Technician_Handbook.pdf
Characters extracted: 3511938

All PDFs loaded successfully!


In [ ]:
import os

print(os.getcwd())
print(os.listdir("/content"))

In [62]:
import fitz

pdf = "/content/amt_general_handbook.pdf"

doc = fitz.open(pdf)

print("Pages:", doc.page_count)

Pages: 698


In [63]:
for doc in all_documents:
    print(doc["filename"])
    print(len(doc["text"]))
    print("-"*60)

amt_powerplant_handbook.pdf
1830419
------------------------------------------------------------
amt_general_handbook.pdf
2381774
------------------------------------------------------------
FAA-H-8083-31B_Aviation_Maintenance_Technician_Handbook.pdf
3511938
------------------------------------------------------------


In [64]:
!pip install -q langchain-text-splitters

In [65]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [66]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ".", " ", ""]
)

In [67]:
chunk_data = []

for document in all_documents:

    chunks = text_splitter.split_text(document["text"])

    for i, chunk in enumerate(chunks):

        chunk_data.append({
            "source": document["filename"],
            "chunk_id": i,
            "text": chunk
        })

print("Total Chunks:", len(chunk_data))

Total Chunks: 9723


In [68]:
print(chunk_data[0])

{'source': 'amt_powerplant_handbook.pdf', 'chunk_id': 0, 'text': 'Aviation Maintenance Technician \nHandbook–Powerplant\n2023\nU.S. Department of Transportation\nFederal Aviation Administration\nFlight Standards Service\nii\nThe Aviation Maintenance Technician Handbook–Powerplant (FAA-H-8083-32B) is one of a series of three handbooks for \npersons preparing for certification as a powerplant mechanic. It is intended that this handbook provide the basic information \non principles, fundamentals, and technical procedures in the subject matter areas relating to the powerplant rating. It is \ndesigned to aid students enrolled in a formal course of instruction, as well as the individual who is studying on his or her \nown. Since the knowledge requirements for the airframe and powerplant ratings closely parallel each other in some subject \nareas, the chapters which discuss fire protection systems and electrical systems contain some material which is also duplicated \nin the Aviation Maintena

In [69]:
import pandas as pd

df = pd.DataFrame(chunk_data)

df.head()

,source,chunk_id,text
0,amt_powerplant_handbook.pdf,0,Aviation Maintenance Technician \nHandbook–Pow...
1,amt_powerplant_handbook.pdf,1,in the Aviation Maintenance Technician Handboo...
2,amt_powerplant_handbook.pdf,2,manufacturer's manuals or other textbooks if m...
3,amt_powerplant_handbook.pdf,3,"Testing Standards Branch, AFS-630, P.O. Box 25..."
4,amt_powerplant_handbook.pdf,4,Teledyne Continental Motors (www.genuinecontin...


In [70]:
df.to_csv("processed_chunks.csv", index=False)

print("Saved successfully!")

Saved successfully!


In [71]:
from google.colab import files

files.download("processed_chunks.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>